# 00a — Fetch Kaggle Data
Downloads all three datasets from Kaggle (no live scraping needed --
FBref blocks scrapers with Cloudflare, so we use a pre-scraped Kaggle
mirror of the same FBref data instead):
- `davidcariboo/player-scores` — market values, appearances, games
- `irrazional/transfermarkt-injuries` — injury records
- `siddhrajthakor/fbref-premier-league-202425-player-stats-dataset` — FBref-sourced PL player stats (2024-25 season)

**Before running this:** you need a free Kaggle account and an API token (`kaggle.json`) placed at `~/.kaggle/kaggle.json` (on Windows: `C:\Users\<you>\.kaggle\kaggle.json`). See https://www.kaggle.com/docs/api if you haven't set this up yet.

**Want a different or additional season?** Kaggle also has `hubertsidorowicz/football-players-stats-2025-2026` (top-5 European leagues, actively maintained, more seasons) if you want extra history for the forecasting step -- just add its slug to the DATASETS dict below.

## Setup

In [1]:
import subprocess
import sys
import zipfile
from pathlib import Path

RAW_DIR = Path('../data/raw')
RAW_DIR.mkdir(parents=True, exist_ok=True)

DATASETS = {
    'player_scores': 'davidcariboo/player-scores',
    'injuries': 'irrazional/transfermarkt-injuries',
    'fbref': 'akshankrithick/fbref-2017-2024-for-europes-top-5-leagues',
}


## Download function
Uses `python -m kaggle` rather than the bare `kaggle` command — more reliable across different OS/PATH setups.

In [2]:
def download_kaggle_dataset(slug: str, dest_subdir: str):
    dest = RAW_DIR / dest_subdir
    dest.mkdir(exist_ok=True)
    if any(dest.glob('*.csv')):
        print(f"Skipping {slug} -- CSVs already present in {dest}")
        return
    print(f'Downloading {slug} -> {dest}')
    subprocess.run(
        [sys.executable, '-m', 'kaggle', 'datasets', 'download',
         '-d', slug, '-p', str(dest), '--force'],
        check=True,
    )
    for zf in dest.glob('*.zip'):
        with zipfile.ZipFile(zf) as z:
            z.extractall(dest)
        zf.unlink()

for name, slug in DATASETS.items():
    download_kaggle_dataset(slug, name)

Skipping davidcariboo/player-scores -- CSVs already present in ..\data\raw\player_scores
Skipping irrazional/transfermarkt-injuries -- CSVs already present in ..\data\raw\injuries
Skipping akshankrithick/fbref-2017-2024-for-europes-top-5-leagues -- CSVs already present in ..\data\raw\fbref


## Run the downloads

In [9]:
for name, slug in DATASETS.items():
    download_kaggle_dataset(slug, name)

Skipping davidcariboo/player-scores -- CSVs already present in ..\data\raw\player_scores
Skipping irrazional/transfermarkt-injuries -- CSVs already present in ..\data\raw\injuries


## Quick check: what did we get?
Run this to see the actual file names and column headers before wiring up the dbt staging models — the real schema might differ slightly from what's assumed in the SQL files.

In [4]:
import pandas as pd
import glob

for f in (glob.glob('../data/raw/player_scores/*.csv') +
          glob.glob('../data/raw/injuries/*.csv') +
          glob.glob('../data/raw/fbref/*.csv')):
    df = pd.read_csv(f, nrows=3)
    print(f)
    print(df.columns.tolist())
    print()

../data/raw/player_scores\appearances.csv
['appearance_id', 'game_id', 'player_id', 'player_club_id', 'player_current_club_id', 'date', 'player_name', 'competition_id', 'yellow_cards', 'red_cards', 'goals', 'assists', 'minutes_played']

../data/raw/player_scores\clubs.csv
['club_id', 'club_code', 'name', 'domestic_competition_id', 'total_market_value', 'squad_size', 'average_age', 'foreigners_number', 'foreigners_percentage', 'national_team_players', 'stadium_name', 'stadium_seats', 'net_transfer_record', 'coach_name', 'last_season', 'filename', 'url']

../data/raw/player_scores\club_games.csv
['game_id', 'club_id', 'own_goals', 'own_position', 'own_manager_name', 'opponent_id', 'opponent_goals', 'opponent_position', 'opponent_manager_name', 'hosting', 'is_win']

../data/raw/player_scores\competitions.csv
['competition_id', 'competition_code', 'name', 'sub_type', 'type', 'country_id', 'country_name', 'domestic_league_code', 'confederation', 'total_clubs', 'url']

../data/raw/player_sco

In [10]:
import pandas as pd, glob

for f in glob.glob('../data/raw/fbref/*.csv'):
    df = pd.read_csv(f, nrows=3)
    print(f)
    print(df.columns.tolist())
    print()

../data/raw/fbref\cleaned_2017-18.csv
['rk', 'player', 'nation', 'pos', 'squad', 'comp', 'age', 'born', 'Matches Played', 'Avg Mins per Match', 'Goals', 'Assists', 'Goals & Assists', 'Non Penalty Goals', 'Penalty Kicks Made', 'Expected Goals', 'Exp NPG', 'Progressive Carries', 'Progressive Passes', 'Goals p 90', 'Assists p 90', 'Tackles attempted', 'Tackles Won', '% Dribbles tackled', 'Shots blocked', 'Passes blocked', 'Interceptions', 'Clearances', 'Errors made', 'Goals Against', 'Goals against p 90', 'Saves', 'Saves %', 'Clean Sheets', '% Clean sheets', '% Penalty saves', 'Passes Completed', 'Passes Attempted', 'Pass completion %', 'Progressive passes distance', '% Short pass completed', '% Medium passes completed', '% Long passes completed', 'Key passes', '1/3', 'Passes into penalty area', 'touches_def_pen', 'Take ons attempted', '% Successful take-ons', 'Times tackled during take-on', 'carries_prgc', 'carries final 3rd', 'carries penalty area', 'Possessions lost', 'Goals Scored', '

In [11]:
import pandas as pd
df = pd.read_csv('../data/raw/fbref/cleaned_2023-24.csv')
print(df['comp'].unique())

<StringArray>
['Premier League', 'Bundesliga', 'Ligue 1', 'La Liga', 'Serie A']
Length: 5, dtype: str


In [12]:
import pandas as pd
import glob

files = glob.glob('../data/raw/fbref/*.csv')
combined = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
combined.to_csv('../data/raw/fbref/combined_all_seasons.csv', index=False)
print(f"Combined {len(files)} files into {len(combined)} rows")

Combined 7 files into 18243 rows
